## 准备数据

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [3]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [4]:
# class myModel:
#     def __init__(self):
#         ####################
#         '''声明模型对应的参数'''
#         ####################
#     def __call__(self, x):
#         ####################
#         '''实现模型函数体，返回未归一化的logits'''
#         ####################
#         return logits
        
# model = myModel()

# optimizer = optimizers.Adam()

class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal(shape=(28 * 28, 128), stddev=0.01), trainable=True)
        self.b1 = tf.Variable(tf.zeros(shape=(128,)), trainable=True)
        self.W2 = tf.Variable(tf.random.normal(shape=(128, 10), stddev=0.01), trainable=True)
        self.b2 = tf.Variable(tf.zeros(shape=(10,)), trainable=True)
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x_flattened = tf.reshape(x, [-1, 784])
        h1 = tf.matmul(x_flattened, self.W1) + self.b1
        activated_h1 = tf.tanh(h1)
        logits = tf.matmul(activated_h1, self.W2) + self.b2
        return logits

model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [5]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.2868428 ; accuracy 0.46433333
epoch 1 : loss 2.286546 ; accuracy 0.46863332
epoch 2 : loss 2.2862484 ; accuracy 0.47301668
epoch 3 : loss 2.2859495 ; accuracy 0.47701666
epoch 4 : loss 2.2856495 ; accuracy 0.48076665
epoch 5 : loss 2.2853482 ; accuracy 0.48446667
epoch 6 : loss 2.2850459 ; accuracy 0.48816666
epoch 7 : loss 2.2847419 ; accuracy 0.49143332
epoch 8 : loss 2.284437 ; accuracy 0.49523333
epoch 9 : loss 2.2841306 ; accuracy 0.49875
epoch 10 : loss 2.283823 ; accuracy 0.50208336
epoch 11 : loss 2.283514 ; accuracy 0.5054833
epoch 12 : loss 2.2832036 ; accuracy 0.50845
epoch 13 : loss 2.2828922 ; accuracy 0.5113
epoch 14 : loss 2.2825792 ; accuracy 0.51411664
epoch 15 : loss 2.2822645 ; accuracy 0.5168833
epoch 16 : loss 2.2819488 ; accuracy 0.5197833
epoch 17 : loss 2.2816315 ; accuracy 0.52281666
epoch 18 : loss 2.2813125 ; accuracy 0.5254833
epoch 19 : loss 2.2809923 ; accuracy 0.5276667
epoch 20 : loss 2.2806704 ; accuracy 0.53045
epoch 21 : loss 2.280346